# 05 - Text + Image Fusion

Goal: combine the text and image models and check if this beats text alone.

1. Turn the text SVM scores into probabilities.
2. Average the two: the text model gets a weight between 0 and 1, the image model gets the rest.
3. Compare text alone, image alone and fusion on the same 16984 validation products.

## 1. Text probabilities

We use the optimised text model from notebook 04 (all features, binary TF-IDF, LinearSVC with C = 1). A LinearSVC gives scores, not probabilities. Calibration learns on the training set (3-fold CV) how to turn these scores into probabilities.

In [1]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC

from src.categories import CATEGORY_NAMES
from src.metrics import compute_per_class_metrics

data = pd.read_csv("../data/processed/text_train_processed.csv")
train = data[data["split"] == "train"]
validation = data[data["split"] == "validation"]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, binary=True)
X_train = vectorizer.fit_transform(train["text_combined"])
X_validation = vectorizer.transform(validation["text_combined"])

# Optimised LinearSVC from notebook 04, wrapped to output probabilities
text_model = CalibratedClassifierCV(LinearSVC(C=1, random_state=42), cv=3)
text_model.fit(X_train, train["prdtypecode"])
classes = text_model.classes_

text_probs = pd.DataFrame(text_model.predict_proba(X_validation), columns=classes)
text_probs.insert(0, "row_id", validation["row_id"].to_numpy())
text_probs.to_csv("../models/outputs/05_text_validation_predictions.csv", index=False)

print(f"Text probabilities saved: {text_probs.shape}")
print(f"Each row sums to 1: {np.allclose(text_probs[classes].sum(axis=1), 1)}")

Text probabilities saved: (16984, 28)
Each row sums to 1: True


## 2. Match text and image on the same products

In [2]:
image = pd.read_csv("../models/outputs/03_7_convnext_tiny_optimized_validation_predictions.csv")
image_probs = image[["row_id"] + [f"prob_{c}" for c in classes]]
image_probs.columns = ["row_id"] + list(classes)

both = (
    validation[["row_id", "prdtypecode"]]
    .merge(text_probs, on="row_id")
    .merge(image_probs, on="row_id", suffixes=("_text", "_image"))
)
y_true = both["prdtypecode"].to_numpy()
P_text = both[[f"{c}_text" for c in classes]].to_numpy()
P_image = both[[f"{c}_image" for c in classes]].to_numpy()

print(f"Products scored by both: {len(both)}")

Products scored by both: 16984


## 3. Choose the text weight

We try text weights from 0 (image only) to 1 (text only), in steps of 0.1.

Limitation: image probabilities exist only for validation, so the weight is chosen on validation. The fusion score is therefore slightly optimistic.

In [3]:
weights = np.round(np.arange(0, 1.01, 0.1), 1)
weight_scores = []
for w in weights:
    fused = w * P_text + (1 - w) * P_image
    pred = classes[fused.argmax(axis=1)]
    weight_scores.append({"w_text": w, "weighted_f1": f1_score(y_true, pred, average="weighted")})

weight_scores = pd.DataFrame(weight_scores)
best_w = weight_scores.loc[weight_scores["weighted_f1"].idxmax(), "w_text"]
display(weight_scores.rename(columns={"w_text": "Text weight", "weighted_f1": "Weighted F1"}).style.hide(axis="index").format(precision=4))
print(f"\nBest text weight: {best_w}")

Text weight,Weighted F1
0.0000,0.6970
0.1000,0.7201
0.2000,0.7523
0.3000,0.7886
0.4000,0.8236
0.5000,0.8524
0.6000,0.8626
0.7000,0.8588
0.8000,0.8542
0.9000,0.8485



Best text weight: 0.6


## 4. Text vs image vs fusion

In [4]:
pred_text = classes[P_text.argmax(axis=1)]
pred_image = classes[P_image.argmax(axis=1)]
pred_fusion = classes[(best_w * P_text + (1 - best_w) * P_image).argmax(axis=1)]

summary = pd.DataFrame({
    "model": ["Text only", "Image only", f"Fusion (w={best_w})"],
    "weighted_f1": [f1_score(y_true, p, average="weighted") for p in (pred_text, pred_image, pred_fusion)],
    "macro_f1": [f1_score(y_true, p, average="macro") for p in (pred_text, pred_image, pred_fusion)],
})
summary.rename(columns={"model": "Model", "weighted_f1": "Weighted F1", "macro_f1": "Macro F1"}).style.hide(axis="index").format(precision=4)

Model,Weighted F1,Macro F1
Text only,0.8430,0.8326
Image only,0.6970,0.6675
Fusion (w=0.6),0.8626,0.8529


In [5]:
text_scores = compute_per_class_metrics(y_true, pred_text, classes)
fusion_scores = compute_per_class_metrics(y_true, pred_fusion, classes)

per_class = pd.DataFrame({
    "code": classes,
    "category": [CATEGORY_NAMES[c] for c in classes],
    "text_f1": text_scores["f1"],
    "fusion_f1": fusion_scores["f1"],
})
per_class["gain"] = per_class["fusion_f1"] - per_class["text_f1"]
per_class.sort_values("gain", ascending=False).rename(columns={
    "code": "Code", "category": "Category", "text_f1": "Text F1", "fusion_f1": "Fusion F1", "gain": "Gain",
}).style.hide(axis="index").format(precision=3)

Code,Category,Text F1,Fusion F1,Gain
2705,New Books,0.711,0.874,0.164
10,Used Books,0.566,0.685,0.119
40,Games,0.753,0.837,0.084
2403,Book Lots,0.768,0.836,0.068
2280,Magazines,0.810,0.870,0.060
1281,Board Games,0.616,0.646,0.030
1140,Figurines,0.830,0.859,0.029
1160,Cards,0.954,0.979,0.025
1180,Wargames,0.697,0.715,0.018
2462,Used Games,0.825,0.840,0.015


### Analysis

- Text alone (calibrated): weighted F1 0.8430. Image alone (`convnext_tiny_optimized`): 0.6970.
- Fusion with 60% text and 40% image: weighted F1 0.8626 (+2.0 points) and macro F1 0.8529 (+2.0 points).
- The weight is stable between 0.5 and 0.7 (0.8524 to 0.8626).
- Book and media categories gain the most: New Books +16.4 points, Used Books +11.9, Games +8.4, Book Lots +6.8, Magazines +6.0. These are the categories where text is weakest.
- No category loses much. The largest losses are Decor and RC Models (-1.1 points).
- Limitation: the weight was chosen on validation, so the gain is slightly optimistic. A clean version needs image predictions on train to choose the weight with cross-validation.

## Conclusion

Combining text and the optimised ConvNeXt-Tiny image model gives the best single-image fusion result: 0.8626 weighted F1 (0.8529 macro F1) on validation, against 0.8430 for text alone.